In [20]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import classification_report

In [ ]:
rf_best = pickle.load(open('exoplanet_classifier.pkl', 'rb'))
realistic_test_v2 = pd.DataFrame({
    'Scenario': [
        'Red Dwarf Habitable (Class 1)',
        'K-type Habitable (Class 2)',
        'Hot Massive Star (Class 0)',
        'Proxima Centauri b',
        'TRAPPIST-1e (Red Dwarf)',
        'Kepler-452b (Sun-like)',
        'Hot Jupiter (A-star)',
        'Super-Earth Red Dwarf',
        'Venus-like',
        'Mars-like'
    ],
    'P_PERIOD': [50, 120, 3000, 11.2, 6.1, 385, 3.5, 25, 5, 500],
    'P_SEMI_MAJOR_AXIS': [0.6, 2.4, 6.7, 0.05, 0.03, 1.05, 0.03, 0.08, 0.05, 3.0],
    'S_DISTANCE': [70, 340, 630, 1.3, 12.1, 430, 80, 25, 50, 200],
    'S_MASS': [0.3, 0.65, 1.0, 0.12, 0.089, 1.04, 1.5, 0.35, 1.2, 0.8],
    'S_RADIUS': [0.5, 0.68, 1.5, 0.14, 0.12, 1.11, 1.8, 0.45, 1.0, 0.7],
    'S_TEMPERATURE': [3340, 4350, 5500, 3042, 2566, 5757, 7500, 3500, 5800, 4000],
    'S_LOG_G': [4.12, 4.58, 4.35, 4.95, 5.09, 4.9, 4.0, 4.3, 4.8, 4.6]
})



feature_cols = ['P_PERIOD', 'P_SEMI_MAJOR_AXIS', 'S_DISTANCE', 
                'S_MASS', 'S_RADIUS', 'S_TEMPERATURE', 'S_LOG_G']

X_test = realistic_test_v2[feature_cols]


# Make predictions
X_realistic = realistic_test_v2[feature_cols]
predictions = rf_best.predict(X_realistic)
probabilities = rf_best.predict_proba(X_realistic)

# Results
results = realistic_test_v2[['Scenario']].copy()
results['Predicted_Class'] = predictions.astype(int)
results['Predicted class probability'] = np.round(probabilities.max(axis=1), 4)
results['Prob_Class_0'] = np.round(probabilities[:, 0], 4)
results['Prob_Class_1'] = np.round(probabilities[:, 1], 4)
results['Prob_Class_2'] = np.round(probabilities[:, 2], 4)

class_labels = {0: 'Non-Habitable', 1: 'Moderate (Red Dwarf)', 2: 'Highly Habitable (K-type)'}
results['Prediction'] = results['Predicted_Class'].map(class_labels)

print("MODEL PREDICTIONS")
print()
print(results[['Scenario', 'Prediction', 'Predicted class probability']].to_string(index=False))



# Analysis
print("\n" + "="*100)
print()
print("INTERPRETATION")

print("""
Class 0 (Non-Habitable):
  → Hot massive stars (5500K), planets in distant orbits (3000 day period)
  → Too extreme for habitability
  
Class 1 (Moderately Habitable):
  → Red dwarf systems (3340K, 0.3 solar masses)
  → Habitable zone around low-mass stars
  → Close orbits (50 day period)
  
Class 2 (Highly Habitable):
  → K-type stars (4350K, 0.65 solar masses)
  → Moderate orbital conditions (120 day period)
  → Most similar to Earth's solar neighborhood
""")


MODEL PREDICTIONS

                     Scenario                Prediction  Confidence
Red Dwarf Habitable (Class 1)      Moderate (Red Dwarf)      0.5359
   K-type Habitable (Class 2) Highly Habitable (K-type)      0.7745
   Hot Massive Star (Class 0)             Non-Habitable      0.8344
           Proxima Centauri b             Non-Habitable      0.7229
      TRAPPIST-1e (Red Dwarf)             Non-Habitable      0.7314
       Kepler-452b (Sun-like)             Non-Habitable      0.5772
         Hot Jupiter (A-star)             Non-Habitable      1.0000
        Super-Earth Red Dwarf Highly Habitable (K-type)      0.5401
                   Venus-like             Non-Habitable      0.9914
                    Mars-like             Non-Habitable      0.5980


INTERPRETATION

Class 0 (Non-Habitable):
  → Hot massive stars (5500K), planets in distant orbits (3000 day period)
  → Too extreme for habitability

Class 1 (Moderately Habitable):
  → Red dwarf systems (3340K, 0.3 solar masses)
 

In [22]:
results.to_csv('predictions_results.csv', index=False)
print("\n Results saved")


 Results saved
